In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Model Parameters ---
vocab_size = 1000  # Size of the vocabulary
block_size = 128   # Maximum sequence length
embed_dim = 256    # Embedding dimension
num_heads = 4      # Number of attention heads
num_layers = 6     # Number of transformer blocks
dropout_rate = 0.1 # Dropout rate


# --- Self-Attention Head ---
class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.dropout = nn.Dropout(dropout_rate)
        # Register block_size as a buffer so it's saved with the model
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape  # Batch, Time (sequence length), Channels (embedding dimension)

        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        # Compute attention scores ("affinities")
        # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        weights = q @ k.transpose(-2, -1) * (C**-0.5) # Scaled dot-product attention

        # Mask future tokens to prevent cheating in decoder-only models
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weights = F.softmax(weights, dim=-1) # (B, T, T)
        weights = self.dropout(weights)

        # Aggregate value vectors
        out = weights @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out


# --- Multi-Head Attention ---
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim) # Projection back to residual stream dimension
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        # Concatenate outputs from all heads along the last dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out) # Apply linear projection
        out = self.dropout(out)
        return out


# --- Feed-Forward Network ---
class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity and another linear layer """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim), # Typically 4*embed_dim in transformers
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim), # Project back to embed_dim
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.net(x)


# --- Transformer Block ---
class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self):
        super().__init__()
        head_size = embed_dim // num_heads
        self.sa = MultiHeadAttention(num_heads, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # Apply MultiHeadAttention with residual connection and LayerNorm
        x = x + self.sa(self.ln1(x)) # LayerNorm before attention
        # Apply FeedForward with residual connection and LayerNorm
        x = x + self.ffwd(self.ln2(x)) # LayerNorm before feed-forward
        return x


# --- Decoder-Only Model ---
class DecoderOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, embed_dim)
        # Positional embeddings to encode the position of tokens in the sequence
        self.position_embedding_table = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.Sequential(*[Block() for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(embed_dim) # Final layer norm
        self.lm_head = nn.Linear(embed_dim, vocab_size) # Linear layer to project to vocab size

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B, T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B, T, embed_dim)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, embed_dim)
        x = tok_emb + pos_emb # (B, T, embed_dim) - combine token and positional embeddings

        x = self.blocks(x) # (B, T, embed_dim) - apply transformer blocks
        x = self.ln_f(x)   # (B, T, embed_dim) - apply final layer norm
        logits = self.lm_head(x) # (B, T, vocab_size) - project to vocabulary logits

        loss = None
        if targets is not None:
            # Reshape for F.cross_entropy: (N, C, ...) where N is batch, C is num classes
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop idx to the last block_size tokens (decoder can only see up to block_size context)
            idx_cond = idx[:, -block_size:]
            # Get the predictions
            logits, _ = self(idx_cond) # Pass through the model
            # Focus only on the last time step
            logits = logits[:, -1, :] # Becomes (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


# --- Model Instantiation and Test ---
if __name__ == '__main__':
    print("Initializing DecoderOnlyModel...")
    model = DecoderOnlyModel()
    print(f"Model has {sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters")

    # Test with a dummy input
    batch_size = 4
    sequence_length = 8
    dummy_input = torch.randint(0, vocab_size, (batch_size, sequence_length))
    dummy_targets = torch.randint(0, vocab_size, (batch_size, sequence_length))

    print(f"Dummy input shape: {dummy_input.shape}")
    print(f"Dummy targets shape: {dummy_targets.shape}")

    logits, loss = model(dummy_input, dummy_targets)
    print(f"Logits shape: {logits.shape}")
    print(f"Loss: {loss.item():.4f}")

    # Test generation
    print("\nTesting text generation...")
    # Start with a simple prompt (e.g., a batch of 1 with a single token)
    prompt = torch.randint(0, vocab_size, (1, 1))
    generated_sequence = model.generate(prompt, max_new_tokens=20)
    print(f"Generated sequence (first 20 tokens):\n{generated_sequence}")
    print("\nModel setup complete. You can now adapt this structure for your learning and exploration tasks.")


Initializing DecoderOnlyModel...
Model has 5.28 M parameters
Dummy input shape: torch.Size([4, 8])
Dummy targets shape: torch.Size([4, 8])
Logits shape: torch.Size([32, 1000])
Loss: 6.9674

Testing text generation...
Generated sequence (first 20 tokens):
tensor([[496, 558, 506,  53, 896, 571, 555, 888, 157, 742, 861, 297, 555, 576,
         814, 477, 456, 891, 448,   8, 734]])

Model setup complete. You can now adapt this structure for your learning and exploration tasks.


In [2]:
import torch

# --- 1. Data Loading and Preprocessing ---
# Example custom text dataset (increased size)
text = """
Oh, how I love to learn about machine learning!
It's truly fascinating to delve into the world of AI.
From neural networks to transformers, the journey is exciting.
Let's train our model to generate some interesting text now.
""" * 10 # Repeat the text 10 times to make it longer

# Get all unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"All unique characters (vocab size): {vocab_size}")
# Create a mapping from characters to integers and integers to characters
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s): # encoder: take a string, output a list of integers
    return [stoi[c] for c in s]

def decode(l): # decoder: take a list of integers, output a string
    return ''.join([itos[i] for i in l])

# Encode the entire text dataset
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Encoded data shape: {data.shape}")

# Split data into training and validation sets
n = int(0.9 * len(data)) # 90% train, 10% val
train_data = data[:n]
val_data = data[n:]

print(f"Train data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")


All unique characters (vocab size): 34
Encoded data shape: torch.Size([2270])
Train data size: 2043
Validation data size: 227


In [3]:
# --- 2. Dataset and DataLoader ---
# Hyperparameters for training
batch_size = 32 # How many independent sequences will we process in parallel?
block_size = 128 # What is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    # Ensure that len(data) - block_size is non-negative
    if len(data) - block_size <= 0:
        # Handle case where data is too small (e.g., return empty batch or raise error)
        # For this example, we'll ensure data is large enough by repeating it.
        # This should ideally be caught during data preparation.
        raise ValueError(f"Dataset split '{split}' is too small to form a batch with block_size={block_size}. "
                         f"Length of data: {len(data)}. Need at least {block_size + 1} tokens.")

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# Test the batch function
xb, yb = get_batch('train')
print(f"Input batch shape: {xb.shape}")   # (batch_size, block_size)
print(f"Target batch shape: {yb.shape}") # (batch_size, block_size)

# Set device for training
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Input batch shape: torch.Size([32, 128])
Target batch shape: torch.Size([32, 128])
Using device: cuda


In [4]:
import torch.optim as optim

# --- 3. Training Setup ---
# Re-initialize the model with the correct vocab_size derived from our custom data
# (If you changed vocab_size in the first cell, you might want to re-run it first)
# Adjust vocab_size if it's different from the initial parameter.
# For this example, we assume the new vocab_size is smaller or equal,
# or the embedding layer might need to be re-initialized if it's larger.
# For simplicity, we are using the model from the first cell, assuming vocab_size is large enough.
# If vocab_size of data is different from model's vocab_size parameter, it will cause an error.
# You might need to update the model parameters directly if the new vocab_size is significantly different.

# Temporarily update model's vocab_size for this training run if it was defined globally.
# In a real scenario, you'd pass vocab_size to the model's constructor.
# For demonstration, we'll just ensure the global vocab_size matches our data.

# To avoid issues with different vocab_size: Re-instantiate the model
global vocab_size # Declare global if modifying a global var
original_vocab_size = vocab_size # Store original if needed
vocab_size = len(chars)

# Re-instantiate the model with the *new* vocab_size
model = DecoderOnlyModel().to(device)
print(f"Model has {sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters (with updated vocab_size)")

optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# --- 4. Training Loop ---
max_iters = 2000 # Number of training iterations
eval_interval = 200

print(f"\nStarting training on {device} for {max_iters} iterations...")
for iter in range(max_iters):
    # Every eval_interval, evaluate loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1: # Also evaluate at the very end
        model.eval() # Set model to evaluation mode
        with torch.no_grad(): # No gradient calculations needed for evaluation
            X_train, Y_train = get_batch('train')
            X_val, Y_val = get_batch('val')
            logits_train, loss_train = model(X_train.to(device), Y_train.to(device))
            logits_val, loss_val = model(X_val.to(device), Y_val.to(device))
            print(f"Step {iter}: train loss {loss_train.item():.4f}, val loss {loss_val.item():.4f}")
        model.train() # Set model back to training mode

    # Get a random batch of data
    xb, yb = get_batch('train')

    # Move data to the device
    xb = xb.to(device)
    yb = yb.to(device)

    # Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True) # Clear gradients
    loss.backward() # Compute gradients
    optimizer.step() # Update model parameters

print("\nTraining complete!")

# --- Generate text after training ---
model.eval() # Set model to evaluation mode for generation
context = torch.tensor(encode("Oh, how I "), dtype=torch.long, device=device).unsqueeze(0)
print("\nGenerated text (starting from 'Oh, how I '):")
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))


Model has 4.78 M parameters (with updated vocab_size)

Starting training on cuda for 2000 iterations...
Step 0: train loss 3.6492, val loss 3.6236
Step 200: train loss 0.0256, val loss 0.0216
Step 400: train loss 0.0232, val loss 0.0198
Step 600: train loss 0.0215, val loss 0.0215
Step 800: train loss 0.0219, val loss 0.0206
Step 1000: train loss 0.0225, val loss 0.0189
Step 1200: train loss 0.0190, val loss 0.0187
Step 1400: train loss 0.0235, val loss 0.0215
Step 1600: train loss 0.0195, val loss 0.0232
Step 1800: train loss 0.0206, val loss 0.0214
Step 1999: train loss 0.0233, val loss 0.0207

Training complete!

Generated text (starting from 'Oh, how I '):
Oh, how I love to learn about machine learning!
It's truly fascinating to delve into the world of AI.
From neu


In [ ]:
context = torch.tensor(encode('machine learning! '), dtype=torch.long, device=device).unsqueeze(0)
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))

In [ ]:
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")